In [1]:
# Step 2: Load all processed CSVs and create a new SQLite database file

import pandas as pd
import sqlite3
import ast

df_cvs = pd.read_csv("../data/processed/cvs_with_skills.csv")
df_offers = pd.read_csv("../data/processed/job_offers.csv")
df_results = pd.read_csv("../data/processed/matching_results.csv")

df_cvs["extracted_skills"] = df_cvs["extracted_skills"].apply(ast.literal_eval)
df_offers["extracted_skills"] = df_offers["extracted_skills"].apply(ast.literal_eval)

conn = sqlite3.connect("../sql/smartrecruit.db")

print("Database connection created")
print(f"CVs: {len(df_cvs)}, Offers: {len(df_offers)}, Results: {len(df_results)}")

Database connection created
CVs: 224, Offers: 18, Results: 4032


In [2]:
# Step 3: Create the cvs table in the database

# convert the skills list back into a simple comma-separated text, since SQL tables store simple values (not Python lists) per cell
df_cvs_sql = df_cvs.copy()
df_cvs_sql["extracted_skills"] = df_cvs_sql["extracted_skills"].apply(lambda x: ", ".join(x))

# keep only the relevant columns for the database (drop the long raw text columns to keep the table clean)
df_cvs_sql = df_cvs_sql[["filename", "extracted_skills", "num_skills_found", "years_experience", "education"]]

# write this table into the SQLite database, replacing it if it already exists
df_cvs_sql.to_sql("cvs", conn, if_exists="replace", index=False)

print("cvs table created")
print(df_cvs_sql.head(3))

cvs table created
                            filename  \
0  Abiral_Pandey_Fullstack_Java.docx   
1              Achyuth Resume_8.docx   
2           Adelina_Erimia_PMP1.docx   

                                    extracted_skills  num_skills_found  \
0  java, core java, spring, spring mvc, hibernate...                48   
1  java, core java, spring, spring boot, spring m...                50   
2  visio, excel, sharepoint, stakeholder, uat, pr...                25   

   years_experience      education  
0               6.0       Bachelor  
1               8.0  Not specified  
2              10.0         Master  


In [3]:
# Step 4: Create the offres table in the database

df_offers_sql = df_offers.copy()
df_offers_sql["extracted_skills"] = df_offers_sql["extracted_skills"].apply(lambda x: ", ".join(x))

# add an offer_id column so we have a clean unique identifier to link to the matches table later
df_offers_sql = df_offers_sql.reset_index().rename(columns={"index": "offer_id"})

df_offers_sql = df_offers_sql[["offer_id", "job_title", "company", "city", "state", "extracted_skills", "num_skills_found", "years_required", "education_required"]]

df_offers_sql.to_sql("offres", conn, if_exists="replace", index=False)

print("offres table created")
print(df_offers_sql.head(3))

offres table created
   offer_id               job_title           company          city state  \
0         0         Project Manager     American Well        Boston    MA   
1         1  Senior Project Manager  Razorfish Health  Philadelphia    PA   
2         2      IT Project Manager  Cross River Bank      Fort Lee    NJ   

                                    extracted_skills  num_skills_found  \
0  git, api, visio, excel, stakeholder, process i...                16   
1  git, visio, business requirement, process impr...                12   
2  api, excel, stakeholder, pmp, risk management,...                10   

   years_required education_required  
0             6.0      Not specified  
1             7.0           Bachelor  
2             NaN      Not specified  


In [4]:
# Step 5: Create the matches table, linking each CV to each job offer with all the computed scores

df_matches_sql = df_results.copy()

# add a lookup to map each job_title back to its offer_id (so we can properly link tables)
title_to_id = dict(zip(df_offers_sql["job_title"], df_offers_sql["offer_id"]))
df_matches_sql["offer_id"] = df_matches_sql["job_title"].map(title_to_id)

df_matches_sql = df_matches_sql[["cv_filename", "offer_id", "job_title", "company", "text_similarity", "skill_overlap", "experience_match", "education_match", "final_score", "compatible", "ml_predicted_compatible", "ml_confidence"]]

df_matches_sql.to_sql("matches", conn, if_exists="replace", index=False)

print("matches table created")
print(f"Total rows: {len(df_matches_sql)}")
print(df_matches_sql.head(3))

matches table created
Total rows: 4032
                         cv_filename  offer_id               job_title  \
0  Abiral_Pandey_Fullstack_Java.docx         0         Project Manager   
1  Abiral_Pandey_Fullstack_Java.docx         1  Senior Project Manager   
2  Abiral_Pandey_Fullstack_Java.docx         2      IT Project Manager   

            company  text_similarity  skill_overlap  experience_match  \
0     American Well            0.016          0.312             1.000   
1  Razorfish Health            0.019          0.333             0.857   
2  Cross River Bank            0.026          0.300             1.000   

   education_match  final_score  compatible  ml_predicted_compatible  \
0              1.0        0.366           0                        0   
1              1.0        0.353           0                        0   
2              1.0        0.365           0                        0   

   ml_confidence  
0       0.017868  
1       0.011939  
2       0.014783  


In [5]:
# Step 6: Run a test SQL query - find the top 5 best matches for Java developers

query = """
SELECT m.cv_filename, m.job_title, m.company, m.final_score, m.compatible
FROM matches m
WHERE m.job_title LIKE '%Java%'
ORDER BY m.final_score DESC
LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print(result)

                  cv_filename              job_title           company  \
0  Sundar_Java_8+ Years..docx  Senior Java Developer  APPS CONSULTANTS   
1                 Nikhil.docx  Senior Java Developer  APPS CONSULTANTS   
2           Nikith Reddy.docx  Senior Java Developer  APPS CONSULTANTS   
3      Amulya Komatineni.docx  Senior Java Developer  APPS CONSULTANTS   
4          chenna kesava.docx  Senior Java Developer  APPS CONSULTANTS   

   final_score  compatible  
0        0.613           1  
1        0.612           1  
2        0.612           1  
3        0.610           1  
4        0.610           1  


In [6]:
# Step 7: Save and close the database connection

conn.commit()
conn.close()

print("Database saved and closed: sql/smartrecruit.db")

Database saved and closed: sql/smartrecruit.db


In [7]:
# Step 1: Export clean CSVs specifically formatted for Power BI import

import sqlite3
conn = sqlite3.connect("../sql/smartrecruit.db")

df_cvs_export = pd.read_sql_query("SELECT * FROM cvs", conn)
df_offers_export = pd.read_sql_query("SELECT * FROM offres", conn)
df_matches_export = pd.read_sql_query("SELECT * FROM matches", conn)

df_cvs_export.to_csv("../dashboard/cvs_for_powerbi.csv", index=False)
df_offers_export.to_csv("../dashboard/offres_for_powerbi.csv", index=False)
df_matches_export.to_csv("../dashboard/matches_for_powerbi.csv", index=False)

conn.close()

print("Exported 3 CSVs to dashboard/ folder:")
print("- cvs_for_powerbi.csv")
print("- offres_for_powerbi.csv")
print("- matches_for_powerbi.csv")

Exported 3 CSVs to dashboard/ folder:
- cvs_for_powerbi.csv
- offres_for_powerbi.csv
- matches_for_powerbi.csv


In [1]:
# Step: Re-export CV data (now with location + university) for Power BI

import pandas as pd

df_cvs = pd.read_csv("../data/processed/cvs_with_skills.csv")

# convert skills list to readable text (same as before)
import ast
df_cvs["extracted_skills"] = df_cvs["extracted_skills"].apply(lambda x: ", ".join(ast.literal_eval(x)) if isinstance(x, str) and x.startswith("[") else x)

# keep the columns we want in Power BI, now including location and university
df_cvs_export = df_cvs[["filename", "extracted_skills", "num_skills_found", "years_experience", "education", "location", "university"]]

df_cvs_export.to_csv("../dashboard/cvs_for_powerbi.csv", index=False)
print("Re-exported cvs_for_powerbi.csv with location and university columns")
print(df_cvs_export.columns.tolist())

Re-exported cvs_for_powerbi.csv with location and university columns
['filename', 'extracted_skills', 'num_skills_found', 'years_experience', 'education', 'location', 'university']
